# Clinical knowledge-informed prediction of recorded diabetes · version 3

**What changed from version 2.** The analysis software embedded in cell 01 is byte-identical to version 2, so a run with the same Colab runtime and the same options keeps the same run identifier and reuses every completed fit on Drive. Version 3 adds (i) a filter for a cosmetic LightGBM warning, (ii) a resume check after cell 04, and (iii) an **EXT** stage between stages 35 and 36 with post hoc extension analyses whose rules were fixed before they were computed (`EXT_PRESPECIFICATION.md`). EXT results are exploratory and never change the primary comparison.

**Run recipe (STANDARD, resuming run 96ccd0057ac85891).** Keep `RUN_MODE = "STANDARD"` and the three `OPTION_OVERRIDES` exactly as they are; changing either creates a new run and refits everything. Then run the notebook top to bottom once per `RUN_STAGE`, in this order: `"EXPLAIN"`, `"SUBGROUPS"`, `"SCREEN"`, `"EXT"`, `"SENSITIVITY"` (the longest, about nine hours; it resumes after a disconnect), and finally `"REPORT"`. `"ALL"` also works but runs SENSITIVITY before EXT and REPORT. If the resume check reports a different run identifier, see its message before fitting anything.

# Clinical knowledge-informed prediction of recorded diabetes

## 00 · Study purpose and proposal map

A health examination is the prediction starting point. A asks whether diabetes is recorded within three years. B asks whether measured HbA1c is elevated at the examination. These are separate endpoints with separate training, metrics and folders. Pages 7–10 supply the clinical penalty. The three-year cutoff, architecture widths and evaluation choices were made for this reanalysis.

**What to look for.** The included traceability CSV maps the rendered proposal to all 40 stages.

**If the stage stops.** Review partial coverage in the protocol if an intended claim requires repeat visits, subtypes, physiology or clinician ratings.

In [ ]:
# 00 · Study purpose and proposal map
print("Objective 1, version 2. STANDARD runs the full-cohort analysis with a secondary matched penalty comparison.")

## 01 · Connect Drive and set paths

Mount your private Drive and locate the four original data files. STANDARD and ALL are already selected. The notebook extracts and verifies its embedded software.

**What to look for.** The path to the verified software and the selected mode appear below.

**If the stage stops.** If mounting or discovery fails, follow the exact directory layout in README_Colab.md.

In [ ]:
# 01 · Connect Drive and set paths
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, shutil, stat, sys, zipfile
from google.colab import drive
drive.mount("/content/drive")
# Edit paths and run controls here.
ROOT = Path("/content/drive/MyDrive/CKINN_OBJ1")
RAW_DIR = ROOT / "data/raw"
RUN_MODE = "STANDARD"   # AUDIT, SMOKE, STANDARD, FULL
RUN_STAGE = "ALL"    # Stage names and prerequisites are in README_Colab.md
DEVICE = "auto"      # auto, cuda or cpu
OPTION_OVERRIDES = {"broader_rich": True, "site_holdout": True, "alt_zero": True}
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import warnings
# LightGBM 4.6 gives NumPy inputs placeholder names (Column_0, ...); the resulting sklearn warning is cosmetic.
# Predictions are identical with and without names (checked in OBJ1_REVIEW.md, section 2).
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier was fitted with feature names")
EXPECTED_RUN = "96ccd0057ac85891"  # the STANDARD run in results-20261123; used only by the resume check after cell 04
# This embedded archive supplies the verified software for the notebook.
_bundled = base64.b64decode((
    '...'
), validate=True)
soft_hash = hashlib.sha256(_bundled).hexdigest()
if soft_hash != "de537355c44c2e686d5e2073ca09443219c75cff14f0d835fbae8b69b1bd7da2":
    raise ValueError("Embedded software integrity check failed. Reopen an original copy of this notebook.")
package = Path("/content/ckinn_software") / soft_hash[:16]
package.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_bundled)) as archive:
    for entry in archive.infolist():
        rel = PurePosixPath(entry.filename.replace(chr(92), "/"))
        target = (package / str(rel)).resolve()
        if ".." in rel.parts or rel.is_absolute() or ":" in str(rel) or not target.is_relative_to(package.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError("Unsafe software ZIP member")
        if not entry.is_dir():
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)
manifest = json.loads((package / "reproducibility_manifest.json").read_text())
for name, expected in manifest["files"].items():
    if hashlib.sha256((package / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Software package integrity error in {name}")
print("Verified package", package, "·", RUN_MODE, RUN_STAGE)

## 02 · Check the environment

Keep Colab's existing numerical libraries when they meet the declared ranges. Install missing dependencies before importing the model code. SHAP is needed only for its optional tree analysis.

**What to look for.** The cell prints package versions and identifies anything installed.

**If the stage stops.** If a loaded dependency changed, restart the Colab runtime and run from 00. If a version is incompatible, correct the named package.

In [ ]:
# 02 · Check the environment
from importlib import metadata
from packaging.requirements import Requirement
import subprocess
before = {dist.metadata["Name"].lower(): dist.version for dist in metadata.distributions() if dist.metadata["Name"]}
reqs = [Requirement(s) for s in (package / "requirements.txt").read_text().splitlines() if s.strip() and not s.startswith("#")]
if OPTION_OVERRIDES.get("tree_shap", False):
    reqs.append(Requirement("shap>=0.45,<1"))
missing = []
for req in reqs:
    try:
        ver = metadata.version(req.name)
        if ver not in req.specifier:
            raise RuntimeError(f"{req.name} {ver} is outside {req.specifier}. Adjust only this package, restart and rerun.")
    except metadata.PackageNotFoundError:
        if req.name == "torch":
            raise RuntimeError("Select a Colab Python runtime with PyTorch installed, then restart from 00.")
        missing.append(str(req))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *missing])
    changed = [n for n, v in before.items() if metadata.version(n) != v]
    if changed:
        raise RuntimeError(f"Existing dependencies changed: {changed}. Restart the runtime and rerun 00 onward.")
print({req.name: metadata.version(req.name) for req in reqs})

## 03 · Imports, seeds and device

Load numerical and modelling functions from the verified package. Set deterministic PyTorch execution and a fixed random seed. The same mathematical definitions are used on CPU and CUDA.

**What to look for.** The fingerprint lists Python, package versions, device and CUDA information.

**If the stage stops.** For CUDA incompatibility, set DEVICE to cpu in 01 and start a new run.

In [ ]:
# 03 · Imports, seeds and device
import inspect
sys.path.insert(0, str(package / "study"))
from IPython.display import display, Image, Markdown
import numpy as np
import pandas as pd
from data import discover, js, csv, R, J, CORE, NONLAB, PLUS, UNITS, allowed, digest
from stats import weights, metrics
from models import Prep, clinical, Net, seed_all
from runner import Study, VARIANTS, env_info
from analysis import external_contrast, figures, subgroups, explain, updating, evidence, export, internal_summary
from stages import GROUPS, TRAIN_STAGES, SELECTION_COLS, METRIC_COLS, support_table, show, test_cases
from verify import run_checks
config = json.loads((package / "config.json").read_text())
seed_all(config["seed"])
print(json.dumps(env_info(), indent=2))

## 04 · Central configuration and budgets

Select the declared budget before interpreting predictions. The configuration contains search ranges, bootstrap counts and optional analyses. The private run identity binds these choices to code, versions and input bytes.

**What to look for.** A run identifier, profile budget and enabled options are printed. AUDIT contains zero cohort fits.

**If the stage stops.** If a setting appears wrong, edit 01 and restart from the top. Only RUN_STAGE can change while preserving the run identity.

In [ ]:
# 04 · Central configuration and budgets
config.update(profile=RUN_MODE, stage=RUN_STAGE, device=DEVICE)
if RUN_MODE not in config["profiles"] or (RUN_STAGE not in ("ALL", "EXT") and RUN_STAGE not in GROUPS):
    raise ValueError("Unknown profile or stage. Check README_Colab.md.")
if set(OPTION_OVERRIDES) - set(config["options"]):
    raise ValueError("Unknown option in cell 01")
config["options"].update(OPTION_OVERRIDES)
opt = config["options"]
st = Study(ROOT, config, paths=discover(RAW_DIR))
def active(number):
    selected = RUN_STAGE == "ALL" or (RUN_STAGE in GROUPS and number in GROUPS[RUN_STAGE])
    if selected and RUN_MODE == "AUDIT" and number in TRAIN_STAGES:
        print("Skipped in AUDIT mode.")
        return False
    return selected
print("Run", st.run, "·", RUN_MODE, "·", st.device)
print(json.dumps({"budget": st.b, "options": opt, "search": config["search"]}, indent=2))
display(pd.read_csv(package / "proposal_traceability.csv").head(8))

## 04b · Check that completed fits will be reused

The run identifier hashes the input files, every file in `study/`, the configuration (profile and options), the device and the Python and package versions. Completed fits are reused only when all of these match.

**What to look for.** `same run: True` for STANDARD. Otherwise the cell lists each field that differs from the frozen manifest of the expected run.

**If the stage stops.** If versions differ, install the listed versions, restart the runtime and rerun from 00; or accept a new run, which refits everything (about 16 hours on CPU for stages 21 to 29).

In [ ]:
# 04b · Check that completed fits will be reused
print("this run:", st.run, "| expected:", EXPECTED_RUN, "| same run:", st.run == EXPECTED_RUN)
if RUN_MODE == "STANDARD" and st.run != EXPECTED_RUN:
    ref = ROOT / "checkpoints" / "STANDARD" / EXPECTED_RUN / "frozen" / "A" / "manifest.json"
    if ref.exists():
        old_identity = json.loads(ref.read_text())["identity"]
        for field in ["data", "code", "config", "device", "runtime", "versions"]:
            a, b = old_identity.get(field), st.identity.get(field)
            if a != b:
                if isinstance(a, dict) and isinstance(b, dict):
                    diff = {k: (a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)}
                    print(f"differs: {field}: (expected, now) {diff}")
                else:
                    print(f"differs: {field}")
        print("Completed fits of the expected run will not be reused unless these match.")
    else:
        print("No frozen manifest of the expected run on this Drive; this is a new run.")

## 05 · Discover and preserve the inputs

Identify each source by SHA-256. Inspect every sheet and ZIP member, validate paths, and cache the harmonized tables privately. This initial pass can take several minutes.

**What to look for.** Four matching identities and stored audit tables appear. Subsequent stages reuse verified caches.

**If the stage stops.** If byte identities differ, check the originals and avoid resaving either workbook in Excel.

In [ ]:
# 05 · Discover and preserve the inputs
if active(5):
    print(pd.Series(st.identity['data']).to_string())
    st.audit()

## 06 · Inspect schemas and provenance

Read the actual column names, workbook sheet inventory, two JSON provenance files and the cross-cycle consistency report. Note the different definition of activity in the two periods.

**What to look for.** You should see RC, Sheet 1 and all five NHANES archive members.

**If the stage stops.** A missing or ambiguous required field stops loading. Compare the source file with its declared version.

In [ ]:
# 06 · Inspect schemas and provenance
if active(6):
    info = json.loads((st.auditpath / 'inventory.json').read_text(encoding='utf8'))
    for sheet in info['workbooks']:
        print(sheet)
    print(info['zip_members'], info['consistency_report'])

## 07 · Audit completeness and coded values

Count participants, original fields, IDs, exact duplicate rows and missing or zero values. Formula text and cached numbers are inspected separately. All raw-feature summaries are saved even though only the first rows are displayed.

**What to look for.** The tables include the actual data dimensions and NAGALA formula-cache and conversion checks.

**If the stage stops.** Investigate a conversion mismatch or unexpected value in the raw audit before fitting.

In [ ]:
# 07 · Audit completeness and coded values
if active(7):
    for f in ['structure', 'raw_features', 'formula_cache', 'unit_checks']:
        show(st.auditpath / (f + '.csv'))

## 08 · Harmonize names, units and sex codes

Map by verified header. Use mmol/L for glucose and lipids, mmHg for pressure and U/L for ALT/AST. In incident cohorts, 1 denotes male after recoding. Treat unmeasured positive-valued inputs as missing before imputation.

**What to look for.** The shared registry shows both original headers and the analytical unit.

**If the stage stops.** Use the raw level counts and protocol to resolve unexpected codes. Never repair headers by column position.

In [ ]:
# 08 · Harmonize names, units and sex codes
if active(8):
    display(pd.DataFrame([dict(input=k, RICH=R[k], NAGALA=J[k], unit=UNITS[k]) for k in PLUS]))

## 09 · Count eligibility step by step

Restrict to adults with usable follow-up, binary event and the operational baseline glycaemia criteria. The broader RICH version keeps source-eligible higher glucose. NHANES uses separate rules.

**What to look for.** The flow table gives N and events remaining after each rule.

**If the stage stops.** Review the exact step if totals diverge. The FPG <6.1 restriction deliberately changes the study population.

In [ ]:
# 09 · Count eligibility step by step
if active(9):
    show(st.auditpath / 'cohort_flow.csv')

## 10 · Understand time and three-year support

A later event is a known control at three years. An event-free person last seen at or before three years has unknown status. Compare site and cohort support, G(3), weight tails and effective sample size.

**What to look for.** The support table shows unsupported sites explicitly and keeps their denominators.

**If the stage stops.** A site without follow-up beyond three years cannot supply a standalone horizon estimate. See the site sensitivity assumptions.

In [ ]:
# 10 · Understand time and three-year support
if active(10):
    display(support_table(st))

## 11 · Lock predictors and development partitions

Allowlist baseline variables. Split RICH by participant, using status and site where feasible. Each training partition gets its own imputer, scaler and model. In AUDIT, display the registry only.

**What to look for.** Scientific profiles persist development row keys and partition indices before fitting.

**If the stage stops.** A duplicate or missing ID stops the process. Investigate its origin before defining a grouping rule.

In [ ]:
# 11 · Lock predictors and development partitions
if active(11):
    print('Primary inputs', allowed(CORE, 'A'), 'Screening', allowed(NONLAB, 'B'))
    if st.mode != 'AUDIT':
        st.split(st.data('RICH'), st.b['outer'], 'A_RICH_nested', st.seed)

## 12 · Learn preprocessing on training rows

At fitting, replace missing values with training medians, append indicators of which original values were missing, then use stored location and positive scale for measurement columns. This prevents test information from entering the model.

**What to look for.** The implementation of Prep.fit and Prep.transform can be inspected below.

**If the stage stops.** An entirely unobserved training feature prevents that candidate from fitting. Examine the fold audit.

In [ ]:
# 12 · Learn preprocessing on training rows
if active(12):
    print(Prep.__doc__ or 'Preprocessor loaded. Imputation and scale are fitted inside each training split.')
    print(inspect.getsource(Prep))

## 13 · Estimate censoring and define known status

Reverse Kaplan–Meier estimates the chance of remaining observable. Cases receive 1/G(T−), controls receive 1/G(3), and unknowns have zero supervised weight. Detections precede censorings when times tie.

**What to look for.** The weight function below operates on original event times and returns labels, weights and G(3).

**If the stage stops.** If G reaches zero at the horizon, inspect the support table and record the unsupported estimate.

In [ ]:
# 13 · Estimate censoring and define known status
if active(13):
    print('A case at exactly year 3 is included. A censoring at exactly year 3 has unknown status.')
    print(inspect.getsource(weights))

## 14 · Check the mathematics independently

Seven artificial participants include a tied detection/withdrawal, a detection at year three, a censoring at year three and a later detection. Expected nonzero weights are 1, 1.5, 2.25 and 2.25. A separate fixture checks the full-N denominator.

**What to look for.** Both named checks must report OK.

**If the stage stops.** Stop affected analyses if either check fails. Confirm the tie convention and the averaging denominator.

In [ ]:
# 14 · Check the mathematics independently
if active(14):
    test_cases(['test_boundaries_ties_hand_weights', 'test_loss_denominator_and_penalty'])

## 15 · Define metrics and calibration

Brier error is the mean weighted squared error across all eligible participants. Rank measures compare known cases and controls. Calibration asks how closely assigned probabilities agree with observed frequencies.

**What to look for.** The metric implementation exposes AUROC, average precision, calibration, confusion-matrix metrics and support.

**If the stage stops.** A missing slope may reflect a constant model or a single known class. Read the support fields alongside the metric.

In [ ]:
# 15 · Define metrics and calibration
if active(15):
    print('The primary Brier denominator is the entire evaluation cohort.')
    print(inspect.getsource(metrics))

## 16 · Build the matched networks

The figure shows the initial eight-variable design. Each branch extracts learned features, then a fusion layer produces one logit. M1 and M3 share the architecture. We apply sigmoid to report probability.

**What to look for.** The diagram labels the three branches, the two loss terms and the default 1,497 trainable parameters.

**If the stage stops.** If the image is missing, check that cell 01 completed software extraction.

In [ ]:
# 16 · Build the matched networks
if active(16):
    display(Image(filename=str(Path(package) / 'architecture.png')))

## 17 · Differentiate the soft penalty

Penalize the wrong sign of the local probability derivative. Increasing FPG and BMI have positive priors in A. Only observed, in-support pairs contribute. A gradient graph is needed because training differentiates the penalty itself.

**What to look for.** Artificial examples verify parameter-gradient changes at positive lambda, missing-mask exclusion and the HDL direction.

**If the stage stops.** Compare the analytical derivative in raw units with a small central difference if a sign check fails.

In [ ]:
# 17 · Differentiate the soft penalty
if active(17):
    test_cases(['test_loss_denominator_and_penalty', 'test_derivative_sign_masks_and_raw_finite_difference'])
    print(inspect.getsource(clinical))

## 18 · Define statistical comparators

Fit LR, RF, boosted trees and a linear SVM using IPCW. Calibrate SVM scores on training-only out-of-fold data. Ridge Cox uses administrative truncation at three years and a training baseline hazard. Include a constant reference.

**What to look for.** The active model list and per-comparator candidate budget are printed.

**If the stage stops.** A named dependency or estimator-weight failure must be resolved before interpreting its comparison.

In [ ]:
# 18 · Define statistical comparators
if active(18):
    print(st.cfg['models'], 'Comparator candidate budget', st.b['comparator_trials'])

## 19 · Tune, stop and checkpoint

Each inner fold learns preprocessing and its own censoring weights. Select by validation Brier, record the stopping epoch, and refit on the outer training partition. Neural checkpoints save optimizer, RNGs and progress at epoch boundaries.

**What to look for.** The budget and private checkpoint directory identify this run.

**If the stage stops.** After interruption, use the identical profile, code and environment and rerun the interrupted stage.

In [ ]:
# 19 · Tune, stop and checkpoint
if active(19):
    print(st.b, 'Resume directory', st.private)

## 20 · Run small software checks

Use artificial data to exercise the split rules, forbidden features, direction penalty, sample-weight response, ZIP safety and exact CPU checkpoint resumption. SMOKE separately uses small real-data samples to exercise the workflow.

**What to look for.** The verification report is written only after the suite succeeds.

**If the stage stops.** Resolve any failed assertion. A passing test supplies software evidence only.

In [ ]:
# 20 · Run small software checks
if active(20):
    js(st.report / 'software_checks.json', run_checks())

## 21 · Nested internal validation in RICH

In each outer training fold, tune each comparator in inner folds. Score the held-out outer participants once and save their predictions privately. The scientific profiles use all eligible RICH participants.

**What to look for.** A fold-by-model table and private OOF predictions are saved. Timings and candidate failures enter the logs.

**If the stage stops.** Resume INTERNAL after reconnection. Review the affected model/fold if all its candidates failed.

In [ ]:
# 21 · Nested internal validation in RICH
if active(21):
    display(st.internal().head())

## 22 · Summarize internal diagnostics

Read the completed outer-fold table. Differences among folds can reflect case mix and censoring support. Folds share training participants and cannot be used as independent experimental replicates.

**What to look for.** The table contains fold identifiers, model names, case/control counts and held-out metrics.

**If the stage stops.** If it is absent, run INTERNAL using the same profile and options.

In [ ]:
# 22 · Summarize internal diagnostics
if active(22):
    show(st.table / 'A' / 'A_RICH_nested.csv')
    display(internal_summary(st))

## 23 · Choose final configurations in RICH

Repeat the declared inner comparison on all eligible development data. Record the winning configuration, selected lambda, epoch count, validation error and threshold. Anchor the matched pair on M3 settings and epoch count. Select its positive lambda among 1, 10 and 100 using only the RICH inner folds.

**What to look for.** One plan per fitted model is saved before any target scoring.

**If the stage stops.** If plans are missing or invalid, inspect the RICH candidate diagnostics and fix that source-stage issue.

In [ ]:
# 23 · Choose final configurations in RICH
if active(23):
    display(st.selection_table('A')[SELECTION_COLS])

## 24 · Fit and freeze the final models

Fit the selected configurations on RICH. The final ensemble, when configured, uses a fixed equal-weight average across final seeds. All model and preprocessing hashes enter the freeze manifest.

**What to look for.** The path to a complete immutable manifest appears after all final fits finish.

**If the stage stops.** Resume FINAL after interruption. A mismatched hash requires recovering the trusted artifact or a fresh run.

In [ ]:
# 24 · Fit and freeze the final models
if active(24):
    print(st.freeze('A'))

## 25 · Score frozen models in NAGALA

Load the frozen source-trained pipelines and apply them to the Japanese cohort. All target probabilities, labels and participant keys stay in the private Drive results. Only aggregate performance is exported.

**What to look for.** A distinct external metrics table appears, alongside private predictions and seed-specific summaries.

**If the stage stops.** Complete FINAL first. A mismatch in the model or participant fingerprint blocks scoring.

In [ ]:
# 25 · Score frozen models in NAGALA
if active(25):
    display(st.score('A')[METRIC_COLS])

## 26 · Compare paired external Brier errors

First subtract M1 error from M3 error for the same NAGALA participants. Then subtract the matched zero-penalty error from the matched positive-penalty error. Each comparison resamples participants jointly and recomputes censoring weights for each draw.

**What to look for.** Both rows display their penalty strengths, Brier differences, interval bounds and valid replicate counts.

**If the stage stops.** A wide or zero-crossing interval is a valid result. Check support when an interval is unavailable.

In [ ]:
# 26 · Compare paired external Brier errors
if active(26):
    display(pd.Series(external_contrast(st)))
    if opt['ablations']:
        display(pd.Series(external_contrast(st, matched=True)))

## 27 · Inspect calibration, ranking and thresholds

Plot assigned probability against observed risk, and draw ROC and precision–recall curves. Apply the threshold chosen on RICH validation predictions. Review sensitivity and specificity at that threshold.

**What to look for.** A three-panel plot and calibration-bin table are saved for the external cohort.

**If the stage stops.** If only one class is supported, treat the corresponding rank measure as unavailable.

In [ ]:
# 27 · Inspect calibration, ranking and thresholds
if active(27):
    figures(st, 'A')
    show(st.fig / 'A' / 'calibration_roc_pr.png')

## 28 · Compare M1–M5 and single component changes

M2 reuses M1 scores. M4 adds training site weights; M5 adds a sex-specific loss-gap penalty. The matched penalty arms share the architecture, epoch count and seeds. The FPG-only, BMI-only and expanded-prior variants use the matched positive strength.

**What to look for.** The ablation table references the same frozen predictions as the external metrics table.

**If the stage stops.** Enable ablations in 01 before starting the scientific profile. Skipped branches have no result.

In [ ]:
# 28 · Compare M1–M5 and single component changes
if active(28):
    if opt['ablations']:
        p = st.table / 'A' / 'external_metrics.csv'
        if not p.exists():
            raise RuntimeError('Complete FINAL and EXTERNAL before ABLATIONS')
        df = pd.read_csv(p)
        if not set(VARIANTS).issubset(df.model):
            raise ValueError('Component fits missing from frozen manifest')
        csv(st.table / 'A' / 'ablations.csv', df[df.model.isin(VARIANTS + ['M1', 'M2', 'M3'])])
        display(pd.read_csv(st.table / 'A' / 'ablations.csv')[METRIC_COLS])
    else:
        print('Component analyses disabled in this run.')

## 29 · Inspect missingness and validation sensitivities

Review eight- versus twelve-feature availability, mask removal, ALT zeros, broader RICH eligibility, site-held-out validation and conditional censoring. Site codes can have too little horizon support.

**What to look for.** Enabled sensitivity runs have distinct table names. The notebook prints each disabled option.

**If the stage stops.** A missing support group stays visible. Enable expensive options in 01 before selecting a run.

In [ ]:
# 29 · Inspect missingness and validation sensitivities
if active(29):
    for option, src, site in [('site_holdout', 'RICH', True), ('broader_rich', 'RICH_broad', False), ('alt_zero', 'RICH_alt0', False)]:
        if opt[option]:
            display(st.internal(source=src, site=site).head())
        else:
            print('Disabled', option)
    print('Site-IPCW sensitivity is fitted in stage 21 when enabled:', opt['site_ipcw'])
    show(st.auditpath / 'feature_set_availability.csv')

## 30 · Explain risk and audit clinical direction

Use source training medians for Integrated Gradients, then perturb observed raw FPG or BMI within training support. Compare the ensemble's analytical raw-input gradient and a central difference. Save individual cases privately.

**What to look for.** Direction tables show eligible denominators, violations, magnitudes, near-flat fractions and numerical checks.

**If the stage stops.** Investigate numerical failures before reporting direction consistency. Attributions and derivatives answer different questions.

In [ ]:
# 30 · Explain risk and audit clinical direction
if active(30):
    if opt['explain']:
        explain(st, 'A')
        show(st.table / 'A' / 'directions.csv')
    else:
        print('Explanations disabled.')

## 31 · Evaluate prespecified subgroups

Evaluate age, sex, RICH site, lower baseline FPG and availability subgroups. Use the same frozen model in NAGALA and participant-disjoint OOF predictions in RICH. Display groups with sparse evidence.

**What to look for.** Rows include N, known cases, controls and unknowns, plus intervals where there are at least 20 cases and 20 controls.

**If the stage stops.** If a group fails the support rule, report its counts without interpreting its performance.

In [ ]:
# 31 · Evaluate prespecified subgroups
if active(31):
    if opt['subgroups']:
        display(subgroups(st, 'A').head())
    else:
        print('Subgroups disabled.')

## 32 · Optional target intercept update

Reserve target participants for updating and disjoint participants for evaluation. Compare the original and intercept-updated predictions on the same evaluation people after reporting the primary frozen results.

**What to look for.** If enabled, a table records support, estimated offset and same-subset results, or an explicit skip reason.

**If the stage stops.** If the reserved participants contain too few outcomes, retain the recorded skip.

In [ ]:
# 32 · Optional target intercept update
if active(32):
    if opt['target_update']:
        display(pd.DataFrame(updating(st)))
    else:
        print('Target updating disabled.')

## 33 · Audit independent NHANES screening eligibility

Select adults with measured HbA1c and the extract's binary no-diagnosis code. Flag a screen at HbA1c ≥6.5%. Primary eligibility depends on verified DIQ010 recoding. Default output is labelled as-coded.

**What to look for.** The stage displays both periods, candidate counts and the evidence status of the binary recode.

**If the stage stops.** Only switch to verified eligibility after obtaining the producer crosswalk. Keep coded sex labels until its mapping is established.

In [ ]:
# 33 · Audit independent NHANES screening eligibility
if active(33):
    show(st.auditpath / 'cohort_flow.csv')
    print('NHANES analysis label', st.tag('B'), 'Producer recode evidence:', opt['producer_recode_evidence'] or 'unavailable')

## 34 · Train and freeze the earlier screening period

Develop independently in the 2017–2020 extract. Use Age, Sex, BMI and SBP, and a BMI-only positive prior. Neither glycaemic measurements nor self-reported diabetes can enter model inputs.

**What to look for.** The screening freeze and selected penalty strengths are displayed.

**If the stage stops.** Select SCREEN to resume. A missing crosswalk blocks only the verified screening analysis.

In [ ]:
# 34 · Train and freeze the earlier screening period
if active(34):
    if opt['screening'] == 'off':
        print('Screening disabled.')
    else:
        print(st.freeze('B'))
        display(st.selection_table('B', frozen=True)[SELECTION_COLS])

## 35 · Evaluate the later screening period

Apply the frozen screening pipelines in 2021–2023. Report ordinary screening metrics, conditional extract-level uncertainty, subgroup support, and the independent BMI direction audit.

**What to look for.** The B directory contains temporal metrics, curves and selected diagnostics. B has no survey weights.

**If the stage stops.** Inspect temporal drift in measurement and coding. Population-representative inference needs keyed survey design variables.

In [ ]:
# 35 · Evaluate the later screening period
if active(35):
    if opt['screening'] == 'off':
        print('Screening disabled.')
    else:
        display(st.score('B')[METRIC_COLS])
        display(pd.Series(external_contrast(st, 'B')))
        figures(st, 'B')
        if opt['subgroups']:
            subgroups(st, 'B')
        if opt['explain']:
            explain(st, 'B')

## EXT · Post hoc extension analyses (exploratory, rules fixed before computation)

These analyses were added after the STANDARD results were seen. Their rules are in `EXT_PRESPECIFICATION.md` (E1 to E11). They read the frozen models and predictions of this run and add new fits only for E9 (penalty-strength path), E10 (training-size curve) and E11 (RICH without sites 14 and 16). New fits are checkpointed like every other fit and resume after a disconnect. Outputs: `results/tables/<mode>/<run>/A/EXT/*.csv` (aggregate only), included in the stage 39 ZIP.

**What to look for.** E2 shows whether the clinical penalty is active at all (wrong-way gradient shares and the penalty relative to the data loss). E3 and E4 compare M1 and M3 with the simple baselines; E9 and E10 show whether any penalty strength or training size changes external error.

**If the stage stops.** Rerun with `RUN_STAGE = "EXT"`; finished parts are restored from checkpoints. Delete nothing.

In [ ]:
# EXT · Post hoc extension analyses
import base64 as _b64, hashlib as _hl, importlib
_ext = _b64.b64decode((
    '...'
), validate=True)
if _hl.sha256(_ext).hexdigest() != "98705506fa9e1d2758ff8078e8b1d48a61e2b1e9a88dd5015d9863f36ae9a95a":
    raise ValueError("Extension module integrity check failed. Reopen an original copy of this notebook.")
_extdir = Path("/content/ckinn_ext")  # outside study/, so the run identity is unchanged
_extdir.mkdir(parents=True, exist_ok=True)
(_extdir / "ckinn_ext.py").write_bytes(_ext)
sys.path.insert(0, str(_extdir))
import ckinn_ext
importlib.reload(ckinn_ext)
if RUN_STAGE in ("ALL", "EXT") and RUN_MODE != "AUDIT":
    print(ckinn_ext.run_all(st))
    for _f in sorted((st.table / "A" / "EXT").glob("*.csv")):
        print(_f.name)
    show(st.table / "A" / "EXT" / "E3_paired_brier_vs_baselines.csv")
    show(st.table / "A" / "EXT" / "E9_lambda_path_expanded.csv")
    show(st.table / "A" / "EXT" / "E10_training_size_summary.csv")
else:
    print("EXT skipped for RUN_STAGE", RUN_STAGE)

## 36 · Collect publication tables and figures

Read available aggregate tables and summarize actual fit budgets and observed run times. Preserve failed attempts and optional-analysis status when preparing a manuscript.

**What to look for.** The cell lists completed tables and saves the fit budget table when fitting was executed.

**If the stage stops.** If an expected artifact is missing, inspect its prerequisite stage and the run options.

In [ ]:
# 36 · Collect publication tables and figures
if active(36):
    for p in sorted(st.table.rglob('*.csv')):
        print(p.relative_to(st.table))
    if (st.log / 'fits.jsonl').exists():
        fits = pd.read_json(st.log / 'fits.jsonl', lines=True)
        csv(st.table / 'fit_budget.csv', fits.groupby(['task', 'model', 'status'], dropna=False).agg(fits=('model', 'size'), seconds=('seconds', 'sum')).reset_index())

## 37 · Generate a narrative from computed values

Read saved differences and intervals and generate a constrained results narrative. Null, unfavorable and unavailable results are handled explicitly.

**What to look for.** A summary and claim-to-evidence table appear in this run's reports directory.

**If the stage stops.** Never fill an absent result with a proposal example or a result from another profile.

In [ ]:
# 37 · Generate a narrative from computed values
if active(37):
    evidence(st)
    show(st.report / 'summary.md')

## 38 · Record the reproducibility manifest

Capture input and code identities, environment, run settings, budgets, available artifacts and frozen hashes. Keep scientific endpoints and software checks distinct.

**What to look for.** The manifest is written alongside the report.

**If the stage stops.** Compare fingerprints if two sessions give different results or seem to reuse the wrong cache.

In [ ]:
# 38 · Record the reproducibility manifest
if active(38):
    evidence(st)
    show(st.report / 'run_manifest.json')

## 39 · Export aggregates and resume later

Create a ZIP of aggregate tables and figures, code and reproducibility metadata. Models, row-level probabilities, individual explanations and source data stay in the private Drive directories.

**What to look for.** The output path identifies this profile's aggregate ZIP. Download it from Drive when needed.

**If the stage stops.** To resume, select the same mode and settings in 01 and a specific RUN_STAGE; run setup and the selected stage.

In [ ]:
# 39 · Export aggregates and resume later
if active(39):
    print(export(st, package))